# 🎓 Diplomatura en Desarrollo de Aplicaciones con LLMs y Modelos Generativos
## Laboratorio 2: Prompt Engineering y Estructuración de Datos

---
### 📌 Encuadre Académico y Esquema de Evaluación
* **Asignatura:** Desarrollo de Aplicaciones con LLMs y Modelos Generativos (UNC).
* **Módulo:** Práctica Guiada — Bloque de Laboratorio (2 horas sincrónicas).
* **Esquema de Evaluación:** Este laboratorio conforma la **Parte 2 y Cierre del Trabajo Práctico 1 (TP1)**. Junto con el Laboratorio 1 (Hiperparámetros y Modelos Locales), integra la primera entrega formal del programa.
* **Criterio de Aprobación:** Ejecutar los 4 bancos de experimentación interactivos y completar las reflexiones estratégicas en los recuadros editables (`✍️ Tu análisis aquí:`) al pie de cada experimento.

---
### 🎯 Objetivos de Aprendizaje Operativo
Al finalizar esta sesión serás capaz de:
1. **Aplicar los Cinco Principios Universales de Prompt Engineering** para transformar requerimientos vagos en instrucciones deterministas, profesionales y reproducibles.
2. **Implementar técnicas de Aprendizaje en Contexto (*In-Context Learning*)** mediante bancos de ejemplos *Few-Shot* para gobernar excepciones y políticas internas sin reentrenar modelos (*fine-tuning*).
3. **Desplegar Razonamiento Guiado (*Chain-of-Thought* / CoT)** en procesos de auditoría y cálculo de múltiples pasos, convirtiendo el texto emitido en memoria de trabajo activa para erradicar saltos lógicos y alucinaciones aritméticas.
4. **Garantizar salidas estructuradas en formato JSON estricto**, eliminando el texto de cortesía o relleno para habilitar la ingesta directa en bases de datos relacionales, CRMs y pipelines de backend corporativo.

---
### ⚙️ Configuración Obligatoria del Entorno en Google Colab
Para garantizar la inferencia acelerada sin costo alguno:
1. En la barra superior de menús, haz clic en **Entorno de ejecución** (*Runtime*).
2. Selecciona **Cambiar tipo de entorno de ejecución** (*Change runtime type*).
3. En **Acelerador de hardware**, verifica que esté seleccionada la opción **GPU T4**.
4. Haz clic en **Guardar**.

---

## 🛠️ Bloque 0: Preparación del Entorno y Motor Local
En este bloque inicial prepararemos el motor de inferencia local de alta velocidad. Utilizaremos **`llama-cpp-python`** compilado con soporte CUDA para ejecutar localmente un modelo de pesos abiertos (*Open Weights*) en formato **GGUF cuantizado a 4-bits**, garantizando costo cero, privacidad estricta y soberanía total sobre los datos procesados.

* **Motor:** `llama-cpp-python` (aceleración completa por GPU / cuBLAS).
* **Modelo Predeterminado:** `Qwen 2.5 3B Instruct` en formato `GGUF (Q4_K_M)` (~2.0 GB de VRAM). Excelente alineación para seguimiento de directivas complejas, estructuración sintáctica JSON y procesamiento fluido en español neutro.

> 💡 **Nota sobre Modelos:** Si bien se recomienda realizar la experimentación inicial con el modelo predeterminado, se invita a los estudiantes a **experimentar libremente con los otros 4 modelos de la lista** (Llama 3.2, Gemma 2, Gemma 4 y DeepSeek) para comparar sus estilos y métricas. Esta exploración adicional es opcional y no forma parte de los criterios de evaluación obligatorios.

> 💡 **Nota sobre Experimentación y Análisis:** Se recomienda copiar y pegar las celdas de "🚀 Ejecutar Experimento" varias veces una debajo de otra antes de relizar el análisis. Al modificar los parámetros (`Temp`, `Top-p`, etc.) en cada copia y ejecutarlas por separado, podrán comparar diferentes resultados visualmente sin perder el anterior, facilitando un análisis mucho más profundo y completo.

In [1]:
#@title 0.1 Instalación silenciosa de dependencias optimizadas para GPU T4 { display-mode: "form" }
import sys

print("⏳ Instalando dependencias con aceleración CUDA precompilada...")

try:
    !CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 -q
    !pip install huggingface_hub ipywidgets pandas -q
    print("✅ Entorno preparado con éxito (llama-cpp-python, huggingface_hub, ipywidgets).")
except Exception as e:
    print(f"❌ Error durante la instalación de paquetes: {e}")

⏳ Instalando dependencias con aceleración CUDA precompilada...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 GB 620.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 72.2 MB/s eta 0:00:00
✅ Entorno preparado con éxito (llama-cpp-python, huggingface_hub, ipywidgets).


In [2]:
#@title 0.2 Selección y Montaje del Modelo en GPU VRAM { run: "auto" }
import os
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

#@markdown Selecciona el modelo que deseas cargar en la GPU T4:
modelo_seleccionado = "Qwen 2.5 3B Instruct (Alibaba)" #@param ["Qwen 2.5 3B Instruct (Alibaba)", "Llama 3.2 3B Instruct (Meta)", "Gemma 2 2B Instruct (Google)", "Gemma 4 E2B Instruct (Experimental)", "DeepSeek-Coder-V2 3B (DeepSeek)"] {run: "auto"}

modelos_disponibles = {
    "Qwen 2.5 3B Instruct (Alibaba)": {
        "repo_id": "Qwen/Qwen2.5-3B-Instruct-GGUF",
        "filename": "qwen2.5-3b-instruct-q4_k_m.gguf"
    },
    "Llama 3.2 3B Instruct (Meta)": {
        "repo_id": "bartowski/Llama-3.2-3B-Instruct-GGUF",
        "filename": "Llama-3.2-3B-Instruct-Q4_K_M.gguf"
    },
    "Gemma 2 2B Instruct (Google)": {
        "repo_id": "bartowski/gemma-2-2b-it-GGUF",
        "filename": "gemma-2-2b-it-Q4_K_M.gguf"
    },
    "Gemma 4 E2B Instruct (Experimental)": {
        "repo_id": "unsloth/gemma-4-E2B-it-GGUF",
        "filename": "gemma-4-E2B-it-Q4_K_M.gguf"
    },
    "DeepSeek-Coder-V2 3B (DeepSeek)": {
        "repo_id": "bartowski/DeepSeek-Coder-V2-Lite-Instruct-GGUF",
        "filename": "DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf"
    }
}

config = modelos_disponibles[modelo_seleccionado]

print(f"⏳ Iniciando proceso para: {modelo_seleccionado}...")

try:
    # Descarga desde Hugging Face
    model_path = hf_hub_download(
        repo_id=config["repo_id"],
        filename=config["filename"],
        local_dir="./modelos"
    )
    print(f"📦 Pesos listos en: {model_path}")

    # Montaje en GPU VRAM
    print("⏳ Cargando el modelo en la memoria VRAM de la GPU T4...")
    llm = Llama(
        model_path=model_path,
        n_gpu_layers=-1,      # Offloading completo a GPU
        n_ctx=4096,           # Ventana de contexto estándar
        verbose=False
    )
    print(f"✅ {modelo_seleccionado} cargado con éxito en GPU.")
except Exception as e:
    print(f"❌ Error al cargar el modelo: {str(e)}")

⏳ Iniciando proceso para: Gemma 4 E2B Instruct (Experimental)...


gemma-4-E2B-it-Q4_K_M.gguf: reconstructing file:   0%|          |  0.00B / 3.11GB            

gemma-4-E2B-it-Q4_K_M.gguf: downloading bytes:           |  0.00B            

📦 Pesos listos en: /content/modelos/gemma-4-E2B-it-Q4_K_M.gguf
⏳ Cargando el modelo en la memoria VRAM de la GPU T4...


llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 512
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 512


✅ Gemma 4 E2B Instruct (Experimental) cargado con éxito en GPU.


In [3]:
#@title 0.3 Función Auxiliar Universal de Inferencia y Telemetría { display-mode: "form" }

import time
import json
from IPython.display import display, Markdown
from llama_cpp.llama_grammar import LlamaGrammar

def validar_y_formatear_json(texto: str):
    """
    Intenta limpiar y parsear un JSON, devolviendo un diagnóstico en Markdown y el JSON formateado.
    """
    try:
        texto_limpio = texto
        if texto_limpio.startswith("```json"):
            texto_limpio = texto_limpio.replace("```json", "").replace("```", "").strip()
        elif texto_limpio.startswith("```"):
            texto_limpio = texto_limpio.replace("```", "").strip()

        parsed = json.loads(texto_limpio)
        # Formateamos el JSON con sangría de 2 espacios
        json_formateado = json.dumps(parsed, indent=2, ensure_ascii=False)
        claves = list(parsed.keys()) if isinstance(parsed, dict) else f"Lista con {len(parsed)} elementos"

        return f"""
> 🟢 **VALIDACIÓN SINTÁCTICA JSON EXITOSA:**
> Objeto deserializado limpiamente por el gestor transaccional.

```json
{json_formateado}
```
"""
    except Exception as json_err:
        return f"""
> 🔴 **FALLA DE PARSEO JSON EN BACKEND (Sistema Caído):**
> *Error:* `{json_err}`
> *Causa funcional:* La salida contiene comentarios de cortesía, etiquetas rotas o comillas inválidas.
"""

def generar_panel_telemetria(params, metricas, diagnostico_json):
    """
    Construye el bloque Markdown con toda la información de la consulta.
    """
    return f"""
---

## ⚙️ Contexto de la Consulta:

### **System Prompt:**
{params['system_prompt']}

### **Usuario:**
{params['prompt']}

<br>

---

## 💬 Respuesta del Modelo:
{metricas['texto']}

<br>

---

## 📊 Métricas de Inferencia:
* 🤖 **Modelo:** `{params['nombre_modelo']}`
* 📏 **Gramática:** `{params['gramatica_activa']}`
* ⏱️ **Tiempo de respuesta:** `{metricas['duracion']:.2f} s`
* 📥 **Tokens de entrada:** `{metricas['tokens_prompt']}`
* 🔢 **Tokens generados:** `{metricas['tokens_completados']}`
* ⚡ **Velocidad:** `{metricas['velocidad']:.1f} tokens/s`
* 🎛️ **Parámetros aplicados:** `Temp={params['temperature']} | Top-p={params['top_p']} | Max={params['max_tokens']}`
{diagnostico_json}

<br>

---
"""

def ejecutar_consulta(prompt: str, system_prompt: str = "Eres un asistente analítico riguroso y conciso.",
                       temperature: float = 0.1, top_p: float = 0.9, max_tokens: int = 512,
                       validar_json: bool = False, grammar: LlamaGrammar = None):
    """
    Ejecuta una consulta sobre el LLM local y despliega el resultado con métricas funcionales.
    """
    t_inicio = time.time()

    # Detección de compatibilidad para modelos que no soportan el rol 'system' (ej. Gemma)
    es_gemma = "gemma" in llm.model_path.lower()

    if es_gemma:
        mensajes = [
            {"role": "user", "content": f"Instrucciones del Sistema: {system_prompt}\n\nTarea/Pregunta: {prompt}"}
        ]
    else:
        mensajes = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]

    try:
        salida = llm.create_chat_completion(
            messages=mensajes,
            temperature=temperature,
            top_p=top_p,
            max_tokens=max_tokens,
            grammar=grammar
        )
    except Exception as err:
        display(Markdown(f"⚠️ **Falla durante la inferencia:** `{err}`"))
        return ""

    t_fin = time.time()

    # Recolección de métricas
    metricas = {
        "duracion": t_fin - t_inicio,
        "texto": salida["choices"][0]["message"]["content"].strip(),
        "tokens_completados": salida["usage"]["completion_tokens"],
        "tokens_prompt": salida["usage"]["prompt_tokens"]
    }
    metricas["velocidad"] = metricas["tokens_completados"] / metricas["duracion"] if metricas["duracion"] > 0 else 0

    # Preparación de parámetros para el panel
    params = {
        "prompt": prompt,
        "system_prompt": system_prompt,
        "temperature": temperature,
        "top_p": top_p,
        "max_tokens": max_tokens,
        "nombre_modelo": llm.model_path.split('/')[-1],
        "gramatica_activa": "✅ Activa (LlamaGrammar)" if grammar else "❌ Inactiva (Free Text)"
    }

    diagnostico_json = validar_y_formatear_json(metricas["texto"]) if validar_json else ""

    display(Markdown(generar_panel_telemetria(params, metricas, diagnostico_json)))
    return metricas["texto"]

print("✅ Motor de inferencia actualizado con compatibilidad para Gemma y refactorizado.")

✅ Motor de inferencia actualizado con compatibilidad para Gemma y refactorizado.


---

## 🔬 Experimento 1: Los Cinco Principios Universales y Control de Directivas (Naive vs. Sistemático)

### 🎯 Objetivo y Modelo Mental
En el caso de negocio de *FinTech Horizon*, el 90% de las fallas operativas no se debían a limitaciones del modelo, sino a **instrucciones ingenuas (*naive*)** sin pliego de condiciones técnicas. Si le pides a un LLM: *"Clasifica este reclamo"*, recibirás una respuesta genérica y conversacional difícil de automatizar.

El **Prompt Engineering Sistemático** aplica cinco principios agnósticos del proveedor:
1. **Dar Dirección (*Give Direction*):** Asignar una *Persona* o rol profesional especializado.
2. **Especificar el Formato (*Specify Format*):** Delimitar exactamente la taxonomía y la estructura de salida.
3. **Proveer Ejemplos (*Provide Examples*):** Demostrar la calibración esperada.
4. **Evaluar la Calidad (*Evaluate Quality*):** Definir criterios de éxito objetivos.
5. **Dividir el Trabajo (*Divide Labor*):** Descomponer procesos en subtareas.

### 📋 Consigna de Exploración:
1. **Paso 1 (System Prompt Ingenuo):** En el formulario inferior, selecciona el modo `Prompt Ingenuo (Naive)` y ejecuta la inferencia. Observa cómo el modelo incluye saludos, texto redundante y un tono no estructurado.
2. **Paso 2 (System Prompt Sistemático):** Cambia al modo `Prompt Sistemático (5 Principios)` y vuelve a ejecutar. Analiza cómo la respuesta adopta un formato tabular limpio, ceñido a una taxonomía cerrada y listo para ser integrado en el software transaccional de atención.

In [4]:
#@title 🎛️ Configuración del Experimento: Comparación de Directivas { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display

# Definición de templates editables
sys_prompt_naive_default = "Eres un Asistente Financiero en FinTech Horizon"

sys_prompt_sist_default = """Actúa como un Oficial Senior de Operaciones y Triaje Financiero en FinTech Horizon. Tu objetivo es procesar incidentes entrantes con precisión técnica y sin texto de cortesía conversacional.

Analiza el siguiente reclamo y clasifícalo estrictamente según estas reglas de negocio:
- Categorías permitidas: [CARGO_DUPLICADO, FRAUDE_POTENCIAL, FALLA_SISTEMA, CANCELACION_SERVICIO, CONSULTA_GENERAL]
- Severidad: [CRITICA, ALTA, MEDIA, BAJA]
- Riesgo de Fuga de Cliente (Churn): [ALTO, MEDIO, BAJO]
- Acción Operativa Inmediata: [Acción sugerida en menos de 10 palabras]

Entrega obligatoriamente la respuesta en formato Markdown con la siguiente estructura tabular:
| Campo | Valor Asignado |
| :--- | :--- |
| **ID Categoría** | (Categoría permitida) |
| **Nivel Severidad** | (Severidad) |
| **Riesgo Churn** | (Riesgo Churn) |
| **Acción Operativa** | (Acción en < 10 palabras) |
"""

user_prompt_default = """Clasifica y analiza este reclamo de un cliente:
\"\"\"
{{texto_evaluar}}
"\"\"
"""

reclamo_cliente_default = "Hola, me cobraron dos veces la suscripción mensual de 45 dólares el día 3 de septiembre. Ya llamé a mi banco y me dijeron que el cobro vino de ustedes. Exijo el reintegro inmediato porque si no voy a dar de baja la cuenta y hacer la denuncia en Defensa del Consumidor."

# Estilo común para alineación
layout_full = widgets.Layout(width='98%', margin='5px 0px 5px 0px')
layout_half = widgets.Layout(width='48%', margin='5px 0px 5px 0px')
style_label = {'description_width': '180px'}

# Widgets de configuración
tipo_estrategia_exp1 = widgets.Dropdown(
    options=["Prompt Ingenuo (Naive)", "Prompt Sistemático (5 Principios)"],
    value="Prompt Ingenuo (Naive)",
    description="Estrategia:",
    style=style_label,
    layout=layout_half
)

temperatura_exp1 = widgets.FloatSlider(
    value=0.1, min=0.0, max=1.0, step=0.05,
    description="Temperature:",
    style=style_label,
    layout=layout_half
)

area_sys_prompt_naive = widgets.Textarea(
    value=sys_prompt_naive_default,
    description="System Prompt Naive:",
    rows=2,
    style=style_label,
    layout=layout_full
)

area_sys_prompt_sistematico = widgets.Textarea(
    value=sys_prompt_sist_default,
    description="System Prompt Sistemático:",
    rows=17,
    style=style_label,
    layout=layout_full
)

reclamo_input_exp1 = widgets.Textarea(
    value=reclamo_cliente_default.strip(),
    description="Texto Reclamo:",
    rows=3,
    style=style_label,
    layout=layout_full
)

area_user_prompt = widgets.Textarea(
    value=user_prompt_default.strip(),
    description="Instrucción de usuario:",
    rows=5,
    style=style_label,
    layout=layout_full
)

print("🎛️ Panel de control avanzado configurado. Edita los prompts y ejecuta la celda inferior:")

# Agrupación y alineación en VBox
ui = widgets.VBox([
    widgets.HTML("<hr><b>🛠️ Edición de System Prompts:</b>"),
    area_sys_prompt_naive,
    area_sys_prompt_sistematico,
    widgets.HTML("<hr><b>🛠️ Edición de User Prompt y texto de reclamo simulado (usa <code>{{texto_evaluar}}</code> como marcador para insertar el texto de reclamo):</b>"),
    area_user_prompt,
    reclamo_input_exp1,
    widgets.HTML("<hr><b>🛠️ Configuración del experimento:</b>"),
    tipo_estrategia_exp1,
    temperatura_exp1
], layout=widgets.Layout(align_items='stretch', padding='10px'))

display(ui)

🎛️ Panel de control avanzado configurado. Edita los prompts y ejecuta la celda inferior:


> 💡 **TIP:** Copiar y pegar la celda "🚀 Ejecutar Experimento" varias veces si deseas comparar distintas ejecuciones manteniendo la configuración o cambiando parámetros sin perder el historial anterior.

In [13]:
#@title 🚀 Ejecutar Experimento { display-mode: "form" }

# 1. Asignación de valores desde widgets
texto_evaluar = reclamo_input_exp1.value
modo_elegido = tipo_estrategia_exp1.value
temp_elegida = temperatura_exp1.value
contenido_template = area_user_prompt.value

# 2. Lógica de selección de estrategia y preparación de prompts
if modo_elegido == "Prompt Ingenuo (Naive)":
    sys_prompt = area_sys_prompt_naive.value
else:
    sys_prompt = area_sys_prompt_sistematico.value

user_prompt = contenido_template.replace("{{texto_evaluar}}", texto_evaluar)

# 3. Inferencia
_ = ejecutar_consulta(
    prompt=user_prompt,
    system_prompt=sys_prompt,
    temperature=temp_elegida,
    top_p=0.9,
    max_tokens=400,
    validar_json=False
)


---

## ⚙️ Contexto de la Consulta:

### **System Prompt:**
Actúa como un Oficial Senior de Operaciones y Triaje Financiero en FinTech Horizon. Tu objetivo es procesar incidentes entrantes con precisión técnica y sin texto de cortesía conversacional.

Analiza el siguiente reclamo y clasifícalo estrictamente según estas reglas de negocio:
- Categorías permitidas: [CARGO_DUPLICADO, FRAUDE_POTENCIAL, FALLA_SISTEMA, CANCELACION_SERVICIO, CONSULTA_GENERAL]
- Severidad: [CRITICA, ALTA, MEDIA, BAJA]
- Riesgo de Fuga de Cliente (Churn): [ALTO, MEDIO, BAJO]
- Acción Operativa Inmediata: [Acción sugerida en menos de 10 palabras]

Entrega obligatoriamente la respuesta en formato Markdown con la siguiente estructura tabular:
| Campo | Valor Asignado |
| :--- | :--- |
| **ID Categoría** | (Categoría permitida) |
| **Nivel Severidad** | (Severidad) |
| **Riesgo Churn** | (Riesgo Churn) |
| **Acción Operativa** | (Acción en < 10 palabras) |


### **Usuario:**
Clasifica y analiza este reclamo de un cliente:
"""
Hola, me cobraron dos veces la suscripción mensual de 45 dólares el día 3 de septiembre. Ya llamé a mi banco y me dijeron que el cobro vino de ustedes. Exijo el reintegro inmediato porque si no voy a dar de baja la cuenta y hacer la denuncia en Defensa del Consumidor.
"""

<br>

---

## 💬 Respuesta del Modelo:
| Campo | Valor Asignado |
| :--- | :--- |
| **ID Categoría** | CARGO_DUPLICADO |
| **Nivel Severidad** | ALTA |
| **Riesgo Churn** | ALTO |
| **Acción Operativa** | Revertir cobro duplicado y notificar al cliente. |

<br>

---

## 📊 Métricas de Inferencia:
* 🤖 **Modelo:** `gemma-4-E2B-it-Q4_K_M.gguf`
* 📏 **Gramática:** `❌ Inactiva (Free Text)`
* ⏱️ **Tiempo de respuesta:** `1.27 s`
* 📥 **Tokens de entrada:** `368`
* 🔢 **Tokens generados:** `79`
* ⚡ **Velocidad:** `62.1 tokens/s`
* 🎛️ **Parámetros aplicados:** `Temp=1.0 | Top-p=0.9 | Max=4000`


<br>

---


### 📝 Registro de Análisis

**Preguntas de reflexión y toma de decisiones:**
1. **Comparación Funcional:** Contrasta la respuesta del caso con *System Prompt Ingenuo* frente a la del *System Prompt Sistemático*. ¿Por qué la primera salida causaría un colapso en un sistema automatizado de tickets (*downstream software*)? La segunda salida, tal como está ¿es una mejor o peor salida estructurada? ¿Esa salida peude ser consumida por el sistema? ¿Qué otra mejora necesita?
2. **Dilución de Atención:** Al aplicar los Principios 1 (Dar Dirección) y 2 (Especificar Formato), ¿qué impacto observas en la concisión de la respuesta y en el consumo de tokens de salida?

> ✍️ **Tu análisis aquí:**
> *(Escribe aquí tus observaciones y conclusiones)*

---

## 🔬 Experimento 2: Aprendizaje en Contexto (*In-Context Learning*): De Zero-Shot a Few-Shot

### 🎯 Objetivo y Modelo Mental
Uno de los mayores dilemas en arquitectura de IA es decidir si un problema requiere reentrenar un modelo (*fine-tuning*) o si puede resolverse mediante **Aprendizaje en Contexto (*In-Context Learning*)**.

Imaginemos que contratamos a un nuevo auditor de riesgos junior para *Fintech Horizon*. El primer día le entregamos una carpeta con 15 casos reales del mes pasado para que revise minuciosamente: 9 aprobados y 6 rechazados, cada uno con su justificación detallada. El nuevo auditor puede empezar a clasificar nuevos casos, ya que su cerebro simplemente **captura el patrón implícito** en los ejemplos y lo replica. El *Few-Shot Prompting* hace exactamente esto con el LLM: le da una "hoja de ruta" de ejemplos para que alinee su criterio al nuestro sin cambiar su arquitectura.

En este experimento evaluamos el triaje de créditos con reglas no convencionales de *FinTech Horizon*:
* *Regla interna:* Los trabajadores independientes con ingresos variables pero historial crediticio intachable y más de 3 años de actividad NO se rechazan: se clasifican como `REVISION_MANUAL_RIESGO`.
* En **Zero-Shot**, el modelo aplica su sesgo estadístico genérico obtenido durante el entrenamiento del modelo y rechaza la solicitud.
* En **Few-Shot**, al observar dos casos resueltos, alinea su criterio inmediatamente a la política interna de la empresa.

### 📋 Consigna de Exploración:
1. **Paso 1 (Modo Zero-Shot):** Configura la estrategia en `Zero-Shot (Sin ejemplos)` y ejecuta varias veces con `Temp` alta. Observa cómo el modelo dictamina diferentes categorías basándose únicamente en el conocimeinto adquirido durante su entrenamiento.
2. **Paso 2 (Modo Few-Shot):** Cambia la estrategia a `Few-Shot (2 Ejemplos de calibración)` y ejecuta nuevamente varias veces sobre el mismo caso, con la misma `Temp`que antes. Analiza cómo el modelo adopta el criterio corporativo exacto (`REVISION_MANUAL_RIESGO`).

> 💡 **TIP:** Copiar y pegar la celda "🚀 Ejecutar Experimento" varias veces si deseas comparar distintas ejecuciones manteniendo la configuración o cambiando parámetros sin perder el historial anterior.

In [6]:
#@title 🎛️ Configuración del Experimento: Zero-Shot vs. Few-Shot { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display

# Definición de templates editables (System Prompts)
sys_prompt_zero_shot_default = """Eres el Analista Principal del Comité de Créditos de FinTech Horizon. Debes clasificar solicitudes de microcrédito exclusivamente en una de estas tres categorías: [APROBADO_DIRECTO, REVISION_MANUAL_RIESGO, RECHAZO_AUTOMATICO]

Responde únicamente con el veredicto y un motivo reglamentario de una oración conforme a la circular 4B."""

sys_prompt_few_shot_default = """Eres el Analista Principal del Comité de Créditos de FinTech Horizon. Debes clasificar solicitudes de microcrédito exclusivamente en una de estas tres categorías: [APROBADO_DIRECTO, REVISION_MANUAL_RIESGO, RECHAZO_AUTOMATICO]

Responde únicamente con el veredicto y un motivo reglamentario de una oración.

Evalúa la solicitud siguiendo exactamente los precedentes y criterios aprobados por el directorio:

[CASO PRECEDENTE 1]
Solicitante: Julián Rossi | Ocupación: Empleado en relación de dependencia | Ingresos: Fijos ($2.500 USD) | Antigüedad: 2 años | Deudas morosas: 0
Dictamen: APROBADO_DIRECTO
Motivo: Ingresos formales estables y sin mora crediticia.

[CASO PRECEDENTE 2]
Solicitante: Marcos Varela | Ocupación: Diseñador Freelance | Ingresos: Variables ($1.900 a $2.800 USD) | Antigüedad: 3.5 años | Deudas morosas: 0
Dictamen: REVISION_MANUAL_RIESGO
Motivo: Conforme a la circular 4B, independientes con más de 3 años y sin mora pasan a revisión manual con extractos bancarios, sin rechazo automático."""

user_prompt_default = """Evalúa la siguiente solicitud de crédito según las políticas estándar de la organización:

> {{solicitud_evaluar}}
"""

solicitud_credito_default = """Solicitante: Laura Morales | Ocupación: Desarrolladora Web Freelance | Ingresos: Variables ($1.200 a $1.500 USD/mes) | Antigüedad laboral: 3 años | Historial deudas morosas: 0 registros negativos."""

# Estilo común para alineación
layout_full = widgets.Layout(width='98%', margin='5px 0px 5px 0px')
layout_half = widgets.Layout(width='48%', margin='5px 0px 5px 0px')
style_label = {'description_width': '120px'}

# Widgets de configuración
estrategia_exp2 = widgets.Dropdown(
    options=["Zero-Shot (Sin ejemplos)", "Few-Shot (2 Ejemplos de calibración)"],
    value="Zero-Shot (Sin ejemplos)",
    description="Estrategia:",
    style=style_label,
    layout=layout_half
)

temperatura_exp2 = widgets.FloatSlider(
    value=0.9, min=0.0, max=1.5, step=0.1,
    description="Temperature:",
    style=style_label,
    layout=layout_half
)

user_input_exp2 = widgets.Textarea(
    value=user_prompt_default.strip(),
    description="Instrucción de usuario:",
    rows=4,
    style=style_label,
    layout=layout_full
)

solicitud_input_exp2 = widgets.Textarea(
    value=solicitud_credito_default.strip(),
    description="Solicitud:",
    rows=3,
    style=style_label,
    layout=layout_full
)

# Nuevos widgets para editar prompts
area_sys_zero_shot = widgets.Textarea(
    value=sys_prompt_zero_shot_default.strip(),
    description="Zero-shot:",
    rows=5,
    style=style_label,
    layout=layout_full
)

area_sys_few_shot = widgets.Textarea(
    value=sys_prompt_few_shot_default.strip(),
    description="Few-shots:",
    rows=17,
    style=style_label,
    layout=layout_full
)

print("🎛️ Panel de control avanzado de In-Context Learning configurado:")

# Agrupación y alineación en VBox (mismo orden que Exp 1)
ui_exp2 = widgets.VBox([
    widgets.HTML("<hr><b>🛠️ Edición de System Prompts:</b>"),
    area_sys_zero_shot,
    area_sys_few_shot,
    widgets.HTML("<hr><b>🛠️ Edición de User Prompt y solicutd simulada (usa <code>{{solicitud_evaluar}}</code> como marcador para insertar la solicitud):</b>"),
    user_input_exp2,
    solicitud_input_exp2,
    widgets.HTML("<hr><b>🛠️ Configuración del experimento:</b>"),
    estrategia_exp2,
    temperatura_exp2
], layout=widgets.Layout(align_items='stretch', padding='10px'))

display(ui_exp2)

🎛️ Panel de control avanzado de In-Context Learning configurado:


In [7]:
#@title 🚀 Ejecutar Experimento { display-mode: "form" }

# 1. Asignación de valores desde widgets
caso_a_evaluar = solicitud_input_exp2.value
modo_in_context = estrategia_exp2.value
temp_in_context = temperatura_exp2.value
user_template = user_input_exp2.value

# 2. Lógica de selección de estrategia (Zero-Shot vs Few-Shot)
user_prompt_final = user_template.replace("{{solicitud_evaluar}}", caso_a_evaluar)

if modo_in_context == "Zero-Shot (Sin ejemplos)":
    sys_prompt_final = area_sys_zero_shot.value
else:
    sys_prompt_final = area_sys_few_shot.value

# 3. Inferencia
_ = ejecutar_consulta(
    prompt=user_prompt_final,
    system_prompt=sys_prompt_final,
    temperature=temp_in_context,
    top_p=0.9,
    max_tokens=150,
    validar_json=False
)


---

## ⚙️ Contexto de la Consulta:

### **System Prompt:**
Eres el Analista Principal del Comité de Créditos de FinTech Horizon. Debes clasificar solicitudes de microcrédito exclusivamente en una de estas tres categorías: [APROBADO_DIRECTO, REVISION_MANUAL_RIESGO, RECHAZO_AUTOMATICO]

Responde únicamente con el veredicto y un motivo reglamentario de una oración conforme a la circular 4B.

### **Usuario:**
Evalúa la siguiente solicitud de crédito según las políticas estándar de la organización:

> Solicitante: Laura Morales | Ocupación: Desarrolladora Web Freelance | Ingresos: Variables ($1.200 a $1.500 USD/mes) | Antigüedad laboral: 3 años | Historial deudas morosas: 0 registros negativos.

<br>

---

## 💬 Respuesta del Modelo:
REVISION_MANUAL_RIESGO. El ingreso variable del solicitante requiere una revisión manual para evaluar la estabilidad del flujo de caja según la circular 4B.

<br>

---

## 📊 Métricas de Inferencia:
* 🤖 **Modelo:** `gemma-4-E2B-it-Q4_K_M.gguf`
* 📏 **Gramática:** `❌ Inactiva (Free Text)`
* ⏱️ **Tiempo de respuesta:** `0.54 s`
* 📥 **Tokens de entrada:** `186`
* 🔢 **Tokens generados:** `34`
* ⚡ **Velocidad:** `62.4 tokens/s`
* 🎛️ **Parámetros aplicados:** `Temp=0.9 | Top-p=0.9 | Max=150`


<br>

---


### 📝 Registro de Análisis

**Preguntas de reflexión y toma de decisiones:**
1. **Efecto de Alineación:** ¿De qué manera la inclusión de dos ejemplos (*Few-Shot*) modificó el dictamen del modelo frente al caso de Laura Morales? ¿Por qué el modelo en *Zero-Shot* podría tender a aceptarlo o rechazarlo?
2. **Trade-offs Operativos:** Compara los tokens de entrada (*Prompts Tokens*) en las métricas de ambas pruebas. Si una empresa procesa 300.000 solicitudes mensuales a través de una API comercial, ¿qué balance económico existe entre inyectar siempre los mismos ejemplos estáticos en cada llamada versus implementar una selección dinámica de ejemplos de forma *inteligente* (por ejemplo, usando RAG)? ¿Sería posible reentrenar un modelo más pequeño que permita embeber en su *memoria* las políticas de solicitudes de la organización?

> ✍️ **Tu análisis aquí:**
> *(Escribe aquí tus observaciones y conclusiones)*

---

## 🔬 Experimento 3: Razonamiento Guiado: Inferencia Directa vs. Chain-of-Thought (CoT)

### 🎯 Objetivo y Modelo Mental
Cuando a un LLM se le exige entregar inmediatamente una cifra o dictamen final en un problema que involucra cálculos o deducciones cruzadas, su tasa de alucinación se dispara. Esto ocurre porque el modelo intenta predecir el token final basándose en patrones estadísticos superficiales, sin haber "computado" realmente los pasos intermedios en su ventana de contexto.

**El pizarrón de trabajo o *Scratchpad*:**
Imagina que estás estás realizando una auditoría frente a un balance lleno de deducciones e impuestos cruzados. Si te prohíben tocar la tiza y te exigen cantar el saldo final en el segundo cero, estás forzado a adivinar: nadie puede resolver mentalmente cuatro operaciones encadenadas en un solo parpadeo.

Al aplicar **Chain-of-Thought (CoT)**, le damos al modelo un "pizarrón". Si se le pide *"Desglosa paso a paso cada deducción"*, el texto que el modelo genera se convierte en su propia **memoria de trabajo activa**. Al escribir el Paso 1, ese resultado queda fijo en su contexto, permitiendo que el Paso 2 se apoye en una base sólida y visible. Cada paso registrado orienta probabilísticamente el paso siguiente hacia la solución correcta, erradicando los saltos lógicos y permitiendo que la respuesta final sea una consecuencia lógica del rastro de pensamiento previo.

En este experimento enfrentamos al modelo a un cálculo de liquidación financiera con comisiones bonificadas e impuestos cruzados.

### 📋 Consigna de Exploración:
1. **Paso 1 (Inferencia Directa):** Selecciona `Inferencia Directa (Respuesta numérica inmediata)` y ejecuta. Observa si el modelo comete un salto lógico o alucina una cifra errónea al saltarse pasos aritméticos.
2. **Paso 2 (Chain-of-Thought Guiado):** Cambia a `Chain-of-Thought (Razonamiento Paso a Paso)` y ejecuta. Analiza cómo el desglose de premisas asegura la exactitud contable y permite auditar en qué paso se originó cada deducción.

> 💡 **TIP:** Copiar y pegar la celda "🚀 Ejecutar Experimento" varias veces si deseas comparar distintas ejecuciones manteniendo la configuración o cambiando parámetros sin perder el historial anterior.

In [8]:
#@title 🎛️ Configuración del Experimento: Razonamiento CoT { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display

problema_financiero_default = "Un cliente realizó una transferencia internacional por 12.500 USD con una comisión base del 2.0%. Por pertenecer al segmento 'Cuenta Corporativa Gold', la plataforma le aplica un descuento del 25% sobre dicha comisión base. Además, el Banco Central exige retener un impuesto regulatorio del 10% aplicado exclusivamente sobre el monto final de la comisión cobrada. ¿Cuál es el costo total definitivo que se le debitará al cliente en concepto de comisiones e impuestos? El separador decimal es la coma y el de miles el punto: 1.000,00"
inferencia_directa_default = "No expliques nada ni muestres cálculos. Entrega únicamente la cifra final con el signo monetario USD."
cot_default = """
Analicemos y auditemos la operación paso a paso antes de dar el veredicto final:
Paso 1: Calcular la comisión base en USD sobre el monto transferido.
Paso 2: Calcular el monto descontado por el beneficio corporativo y la comisión neta resultante.
Paso 3: Calcular la retención del impuesto regulatorio sobre dicha comisión neta.
Paso 4: Sumar la comisión neta y el impuesto para determinar el debito total final.
Conclusión: Indicar la cifra final debitada al cliente.
"""

# Estilo común para alineación
layout_full = widgets.Layout(width='98%', margin='5px 0px 5px 0px')
layout_half = widgets.Layout(width='48%', margin='5px 0px 5px 0px')
style_label = {'description_width': '120px'}

estrategia_cot = widgets.Dropdown(
    options=["Inferencia Directa (Sin razonamiento intermedio)", "Chain-of-Thought (Desglose paso a paso)"],
    value="Inferencia Directa (Sin razonamiento intermedio)",
    description="Modo CoT:",
    style=style_label,
    layout=layout_half
)

temp_cot = widgets.FloatSlider(
    value=0.0, min=0.0, max=0.5, step=0.05,
    description="Temperature:",
    style=style_label,
    layout=layout_half
)

inferencia_directa_input = widgets.Textarea(
    value=inferencia_directa_default.strip(),
    description="Inferencia Directa :",
    rows=2,
    style=style_label,
    layout=layout_full
)

cot_input = widgets.Textarea(
    value=cot_default.strip(),
    description="Chain-of-Thought:",
    rows=7,
    style=style_label,
    layout=layout_full
)

problema_input = widgets.Textarea(
    value=problema_financiero_default.strip(),
    description="Problema:",
    rows=4,
    style=style_label,
    layout=layout_full
)

print("🎛️ Panel de control de razonamiento listo. Selecciona la modalidad y ejecuta:")
ui_exp3 = widgets.VBox([
      widgets.HTML("<hr><b>🛠️ Editar Modo de Razonamiento:</b>"),
      inferencia_directa_input,
      cot_input,
      widgets.HTML("<hr><b>🛠️ Configuración del experimento:</b>"),
      estrategia_cot,
      temp_cot,
      problema_input
    ],
    layout=widgets.Layout(align_items='stretch', padding='10px'))
display(ui_exp3)

🎛️ Panel de control de razonamiento listo. Selecciona la modalidad y ejecuta:


In [9]:
#@title 🚀 Ejecutar Experimento { display-mode: "form" }

# 1. Asignación de valores desde widgets
problema_texto = problema_input.value
modo_seleccionado = estrategia_cot.value
temperatura_seleccionada = temp_cot.value

# 2. Lógica de selección de estrategia (Directa vs CoT)
sys_prompt_cot = f"""Eres un perito liquidador contable y auditor financiero de máxima precisión.\n\n"""

if modo_seleccionado == "Inferencia Directa (Sin razonamiento intermedio)":
    sys_prompt_cot += inferencia_directa_input.value
    max_tok = 80
else:
    sys_prompt_cot += cot_input.value
    max_tok = 450

# 3. Inferencia
_ = ejecutar_consulta(
    prompt=problema_texto,
    system_prompt=sys_prompt_cot,
    temperature=temperatura_seleccionada,
    top_p=0.9,
    max_tokens=max_tok,
    validar_json=False
)


---

## ⚙️ Contexto de la Consulta:

### **System Prompt:**
Eres un perito liquidador contable y auditor financiero de máxima precisión.

No expliques nada ni muestres cálculos. Entrega únicamente la cifra final con el signo monetario USD.

### **Usuario:**
Un cliente realizó una transferencia internacional por 12.500 USD con una comisión base del 2.0%. Por pertenecer al segmento 'Cuenta Corporativa Gold', la plataforma le aplica un descuento del 25% sobre dicha comisión base. Además, el Banco Central exige retener un impuesto regulatorio del 10% aplicado exclusivamente sobre el monto final de la comisión cobrada. ¿Cuál es el costo total definitivo que se le debitará al cliente en concepto de comisiones e impuestos? El separador decimal es la coma y el de miles el punto: 1.000,00

<br>

---

## 💬 Respuesta del Modelo:
125,00

<br>

---

## 📊 Métricas de Inferencia:
* 🤖 **Modelo:** `gemma-4-E2B-it-Q4_K_M.gguf`
* 📏 **Gramática:** `❌ Inactiva (Free Text)`
* ⏱️ **Tiempo de respuesta:** `0.18 s`
* 📥 **Tokens de entrada:** `187`
* 🔢 **Tokens generados:** `6`
* ⚡ **Velocidad:** `32.5 tokens/s`
* 🎛️ **Parámetros aplicados:** `Temp=0.0 | Top-p=0.9 | Max=80`


<br>

---


### 📝 Registro de Análisis

**Preguntas de reflexión y toma de decisiones:**
1. **Auditoría y Explicabilidad:** Comprueba matemáticamente la solución: *Comisión base: 12.500 * 2% = 250. Bonificación 25%: 250 * 0.75 = 187.50. Impuesto 10%: 187.50 * 10% = 18.75. Total debitado: 206.25 USD*. ¿Qué resultado arrojó la inferencia directa frente al enfoque CoT? ¿Por qué el desglose secuencial previene errores aritméticos?
2. **Costo vs. Trazabilidad:** Observa el aumento en los tokens generados (*Output Tokens*) y el tiempo de respuesta. En sectores altamente regulados (finanzas, salud o legal), ¿cómo se justifica el sobrecosto de CoT ante una auditoría externa?

> ✍️ **Tu análisis aquí:**
> *(Escribe aquí tus observaciones y conclusiones)*

---

## 🔬 Experimento 4: Estructuración Garantizada: Extracción Robusta a JSON y Mitigación de Texto Conversacional

### 🎯 Objetivo y Modelo Mental
El principal factor de rotura en arquitecturas con LLMs es la **incompatibilidad sintáctica**: el modelo genera prosa fluida con frases como *"¡Hola! Por supuesto, aquí tienes los datos solicitados: {...}"*, lo cual destruye el parser `json.loads()` de la aplicación transaccional en producción, provocando caídas de servicio.

**Formulario estructurado vs. la carta informal:** Si solicitas a un cliente que redacte libremente su patrimonio, obtendrás omisiones y coloquialismos. Si le entregas un formulario preimpreso con casilleros rígidos, la información ingresa con estructura 100% predecible.

En este experimento procesamos transcripciones caóticas de llamadas telefónicas de soporte para extraer un registro estructurado que alimente directamente la base de datos SQL de *FinTech Horizon*.

### 📋 Consigna de Exploración:
1. **Paso 1 (Prompting Libre):** Selecciona `Extracción Libre (Sin Esquema Rígido)`. Observa cómo el modelo introduce textos de cortesía que provocan la falla del parser (marcador rojo 🔴).
2. **Paso 2 (Esquema por Prompting):** Cambia a `Esquema JSON (solo Prompting)`. Aquí el modelo recibe el esquema como texto. A veces funciona, pero ante variaciones de temperatura sigue siendo propenso a errores de formato o markdown.
3. **Paso 3 (Gramática Estricta):** Selecciona `Gramática JSON Estricta (Nivel Motor/Sampling)`. Comprueba cómo el modelo emite exclusivamente el objeto, activando la bandera verde de validación sintáctica 🟢.

In [10]:
#@title 🎛️ Configuración del Experimento: Estructuración JSON { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display
import json

# --- Valores por defecto ---
transcripcion_soporte_default = """Operador: FinTech Horizon, le atiende Martín, ¿en qué puedo ayudarle?
Cliente: Hola Martín, mirá, estoy desesperado. Soy Carlos Gómez, DNI 28.441.902, titular de la cuenta 4002-9918. Recién a las 14:15 quise sacar plata del cajero de la sucursal Centro y la máquina se trabó: no me dio los 80.000 pesos pero sí me los debitó del saldo. Me tiró un papel con código de error ERR-ATM-992. Necesito esa plata ya para pagar el alquiler hoy mismo."""

esquema_default = {
    "type": "object",
    "properties": {
        "cliente_nombre": {"type": "string"},
        "documento_id": {"type": "string"},
        "monto_reclamado": {"type": "number"},
        "moneda": {"type": "string"},
        "codigo_incidente": {"type": "string"},
        "urgencia_critica": {"type": "boolean"}
    },
    "required": ["cliente_nombre", "documento_id", "monto_reclamado"]
}

sys_prompt_free_default = "Eres un asistente de soporte técnico que extrae datos de clientes."

sys_prompt_prompting_default = """Eres un microservicio backend de ingesta transaccional.
Tu única función es transformar texto no estructurado en objetos JSON estrictamente válidos.
PROHIBIDO incluir saludos o explicaciones. Responde solo el JSON."""

sys_prompt_grammar_default = "Extrae los datos de la llamada en formato JSON."

user_template_default = """Extrae la información de la siguiente transcripción:\n\n\"\"\"\n{{transcripcion}}\n\"\"\""""

# --- Widgets ---
layout_full = widgets.Layout(width='98%', margin='5px 0px 5px 0px')
layout_half = widgets.Layout(width='48%', margin='5px 0px 5px 0px')
style_label = {'description_width': '220px'}

modo_extraccion_exp4 = widgets.Dropdown(
    options=[
        "Extracción Libre (Sin Esquema Rígido)",
        "Esquema JSON (solo Prompting)",
        "Gramática JSON Estricta (Nivel Motor/Sampling)"
    ],
    value="Extracción Libre (Sin Esquema Rígido)",
    description="Estructura:",
    style=style_label,
    layout=layout_half
)

temperatura_exp4 = widgets.FloatSlider(
    value=0.0, min=0.0, max=0.7, step=0.05,
    description="Temperature:",
    style=style_label,
    layout=layout_half
)

area_esquema = widgets.Textarea(
    value=json.dumps(esquema_default, indent=2),
    description="Esquema JSON:",
    rows=29, style=style_label, layout=layout_full
)

area_sys_free = widgets.Textarea(
    value=sys_prompt_free_default,
    description="System Prompt (Libre):",
    rows=2, style=style_label, layout=layout_full
)

area_sys_prompting = widgets.Textarea(
    value=sys_prompt_prompting_default,
    description="System Prompt (Prompting):",
    rows=4, style=style_label, layout=layout_full
)

area_sys_grammar = widgets.Textarea(
    value=sys_prompt_grammar_default,
    description="System Prompt (Grammar):",
    rows=2, style=style_label, layout=layout_full
)

area_user_template = widgets.Textarea(
    value=user_template_default,
    description="Template Usuario:",
    rows=6, style=style_label, layout=layout_full
)

transcripcion_input_exp4 = widgets.Textarea(
    value=transcripcion_soporte_default.strip(),
    description="Transcripción:",
    rows=5, style=style_label, layout=layout_full
)

print("🎛️ Panel de estructuración refactorizado. Edita los componentes y ejecuta:")
ui_exp4 = widgets.VBox([
      widgets.HTML("<hr><b>🛠️ Configuración de System Prompts:</b>"),
      area_sys_free,
      area_sys_prompting,
      area_sys_grammar,
      widgets.HTML("<hr><b>🛠️ Configuración de User Prompt y Schema JSON:</b>"),
      area_user_template,
      area_esquema,
      widgets.HTML("<hr><b>🛠️ Entrada y Control:</b>"),
      modo_extraccion_exp4,
      temperatura_exp4,
      transcripcion_input_exp4
    ],
    layout=widgets.Layout(align_items='stretch', padding='10px'))
display(ui_exp4)

🎛️ Panel de estructuración refactorizado. Edita los componentes y ejecuta:


> 💡 **TIP:** Copiar y pegar la celda "🚀 Ejecutar Experimento" varias veces si deseas comparar distintas ejecuciones manteniendo la configuración o cambiando parámetros sin perder el historial anterior.

In [11]:
#@title 🚀 Ejecutar Experimento { display-mode: "form" }
from llama_cpp.llama_grammar import LlamaGrammar
import json

# 1. Asignación de valores desde widgets
modo = modo_extraccion_exp4.value
temp = temperatura_exp4.value
transcripcion = transcripcion_input_exp4.value
esquema_str = area_esquema.value
user_prompt = area_user_template.value.replace("{{transcripcion}}", transcripcion)

gramatica_a_usar = None

# 2. Lógica de selección de estrategia
if modo == "Extracción Libre (Sin Esquema Rígido)":
    sys_final = area_sys_free.value
    validar = True

elif modo == "Esquema JSON (solo Prompting)":
    sys_final = area_sys_prompting.value
    validar = True

else: # Gramática JSON Estricta
    sys_final = area_sys_grammar.value
    gramatica_a_usar = LlamaGrammar.from_json_schema(esquema_str)
    validar = True

# 3. Inferencia
_ = ejecutar_consulta(
    prompt=user_prompt,
    system_prompt=sys_final,
    temperature=temp,
    top_p=0.9,
    max_tokens=500,
    validar_json=validar,
    grammar=gramatica_a_usar
)


---

## ⚙️ Contexto de la Consulta:

### **System Prompt:**
Eres un asistente de soporte técnico que extrae datos de clientes.

### **Usuario:**
Extrae la información de la siguiente transcripción:

"""
Operador: FinTech Horizon, le atiende Martín, ¿en qué puedo ayudarle?
Cliente: Hola Martín, mirá, estoy desesperado. Soy Carlos Gómez, DNI 28.441.902, titular de la cuenta 4002-9918. Recién a las 14:15 quise sacar plata del cajero de la sucursal Centro y la máquina se trabó: no me dio los 80.000 pesos pero sí me los debitó del saldo. Me tiró un papel con código de error ERR-ATM-992. Necesito esa plata ya para pagar el alquiler hoy mismo.
"""

<br>

---

## 💬 Respuesta del Modelo:
Aquí tienes la información extraída de la transcripción:

* **Nombre del Cliente:** Carlos Gómez
* **DNI:** 28.441.902
* **Número de Cuenta:** 4002-9918
* **Problema Reportado:** El cajero se trabó al intentar retirar dinero.
* **Fecha y Hora del Incidente:** Recién a las 14:15
* **Ubicación del Incidente:** Cajero de la sucursal Centro
* **Detalle del Error:** No se le entregaron los $80.000 pesos, pero se debitó el saldo.
* **Código de Error:** ERR-ATM-992
* **Motivo Urgente:** Necesita el dinero para pagar el alquiler hoy mismo.

<br>

---

## 📊 Métricas de Inferencia:
* 🤖 **Modelo:** `gemma-4-E2B-it-Q4_K_M.gguf`
* 📏 **Gramática:** `❌ Inactiva (Free Text)`
* ⏱️ **Tiempo de respuesta:** `2.15 s`
* 📥 **Tokens de entrada:** `195`
* 🔢 **Tokens generados:** `174`
* ⚡ **Velocidad:** `81.1 tokens/s`
* 🎛️ **Parámetros aplicados:** `Temp=0.0 | Top-p=0.9 | Max=500`

> 🔴 **FALLA DE PARSEO JSON EN BACKEND (Sistema Caído):**
> *Error:* `Expecting value: line 1 column 1 (char 0)`
> *Causa funcional:* La salida contiene comentarios de cortesía, etiquetas rotas o comillas inválidas.


<br>

---


### 📝 Registro de Análisis

**Preguntas de reflexión y toma de decisiones:**
1. **Fragilidad de Integración:** ¿Qué ocurrió en el parser sintáctico cuando ejecutaste la *Extracción Libre*? Si un pipeline transaccional procesa 5.000 reclamos por hora, ¿cuál es el riesgo operativo de que el modelo emita frases como *"Aquí tienes el JSON solicitado:"* antes de las llaves?
2. **JSON vs. YAML:** En entornos masivos donde el costo de tokens de entrada/salida es crítico, ¿qué ventajas ofrece evaluar formatos más ligeros como YAML frente a JSON, y qué riesgo introduce la sensibilidad a los espacios e indentaciones?

> ✍️ **Tu análisis aquí:**
> *(Escribe aquí tus observaciones y conclusiones)*

---

## 🏁 Cierre del Laboratorio y Retrospectiva de Trabajo Práctico

---
### 🧠 Matriz Maestra de Decisión Estratégica en Prompt Engineering

| Técnica de Prompting | Cuándo Aplicarla en Negocio | Parámetros Recomendados | Principal Trade-off / Riesgo |
| :--- | :--- | :--- | :--- |
| **Zero-Shot Sistemático** | Tareas estandarizadas, clasificación genérica y resumen de alto volumen. | Temp: 0.0 - 0.2, Top-p: 0.9 | Puede fallar en excepciones o reglas no estándar de la empresa. |
| **Few-Shot Prompting** | Alineación con criterios de negocio propios, taxonomías internas y políticas corporativas. | Temp: 0.0 - 0.1, Top-p: 0.8 | Incrementa el consumo de tokens de entrada en cada llamada. |
| **Chain-of-Thought (CoT)** | Auditoría contable, evaluación de riesgos, deducción legal y cálculo en múltiples pasos. | Temp: 0.0, Max_Tokens: > 350 | Duplica la latencia y eleva el costo de tokens de salida generados. |
| **Esquema Estricto JSON** | Conexión directa con bases de datos SQL, sistemas ERP, CRMs o microservicios backend. | Temp: 0.0, Top-p: 0.1 - 0.5 | Suprime matices cualitativos no contemplados en el esquema previo. |

---
### ✅ Checklist de Verificación para la Entrega Final del TP1
Antes de descargar o entregar tu cuaderno, verifica los siguientes puntos clave:
* [ ] **Laboratorio 1 completado:** Las celdas de experimentación con hiperparámetros (*Temperature*, *Top-p*, *Max Tokens*, *System Prompt*) fueron ejecutadas y analizadas.
* [ ] **Laboratorio 2 ejecutado:** Se completaron los cuatro experimentos de esta sesión (Principios de Prompting, Few-Shot, CoT y JSON Estructurado).
* [ ] **Análisis funcional completado:** Se completaron con argumentos propios los cuatro recuadros editables `✍️ Tu análisis aquí:` de este cuaderno.
* [ ] **Validación sintáctica:** El Experimento 4 demostró la obtención del marcador verde 🟢 de deserialización JSON.
* [ ] **Respaldo:** Guardaste tu cuaderno mediante `Archivo` -> `Guardar una copia en Drive` o descargaste el archivo `.ipynb` para subirlo a la plataforma de la UNC.

---
### 🌉 Puente Conceptual hacia la Clase 3: Sistemas RAG y Consulta Documental Fundamentada
A lo largo de los Laboratorios 1 y 2 logramos gobernar la estocasticidad del modelo, calibrar sus respuestas mediante ejemplos y forzar esquemas de datos estructurados para backend.

Sin embargo, un LLM aislado sigue teniendo una limitación insalvable: **su memoria es estática y no conoce los manuales internos, políticas actualizadas ni contratos privados de tu organización**. En la **Clase 3**, daremos el salto hacia los **Sistemas RAG (*Retrieval-Augmented Generation*)**: aprenderemos a segmentar documentos corporativos (*chunking*), transformarlos en vectores semánticos (*embeddings*) y conectar el modelo a una base de conocimiento viva para erradicar definitivamente las alucinaciones mediante citas explícitas de fuentes documentales.